In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Copy from Drive to local Colab storage (faster I/O during training)
!cp -r /content/drive/MyDrive/CS273P-LTN-LLM /content/CS273P-LTN-LLM

%cd /content/CS273P-LTN-LLM

Mounted at /content/drive
/content/CS273P-LTN-LLM


In [2]:
!pip install -q accelerate bitsandbytes peft transformers sentencepiece
!pip install -q LTNtorch pysdd scikit-learn scipy tqdm wandb datasets evaluate pandas
!pip install -q python-dotenv beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.3/502.3 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login
login()  # will prompt for your HF token interactively

In [4]:
import os
required = [
    "data/beliefbank/constraints_v2.json",
    "data/beliefbank/calibration_facts.json",
    "data/beliefbank/silver_facts.json",
    "data/beliefbank/beliefbank_augmented.json",
    "data/beliefbank/templates.json",
    "data/beliefbank/non_countable.txt",
]
for f in required:
    print(f"{'OK' if os.path.exists(f) else 'MISSING':>7}  {f}")

     OK  data/beliefbank/constraints_v2.json
     OK  data/beliefbank/calibration_facts.json
     OK  data/beliefbank/silver_facts.json
     OK  data/beliefbank/beliefbank_augmented.json
     OK  data/beliefbank/templates.json
     OK  data/beliefbank/non_countable.txt


In [11]:
# LTN training
!python run.py --config configs/train_llama32_1b_ltn.json

[-] Running in single instance...
[-] Loading decoder only model with quantization
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:00<00:00, 193.21it/s, Materializing param=model.norm.weight]
trainable params: 5636096 || all params: 754911232 || trainables%: 0.746590560729662
[-] Running trainer...
[-] Model: meta-llama/Llama-3.2-1B
[-] Learning rate: 0.0001
[-] Batch size: 4
[-] Accumulation steps: 4
[!] Setting checkpoint save path to: ltn_Llama-3.2-1B
[!] Setting outputs log to: outputs_20260318-173046_Llama-3.2-1B.log
[+] Training in combined mode.
[!] Subsampled training rules: 2000 / 23394
[-] Evaluation @ epoch 0...

[-] Computing scores on calibration split in val mode.
[-] Using prompt format: 0

[-] Querying facts for prompt 0...
  0% 0/20 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  5% 1/20 [00:01<00:29,  1.57s/it]Setting 

In [19]:
# SDD training
!python run.py --config configs/train_llama32_1b_sdd.json

[-] Running in single instance...
[-] Loading decoder only model with quantization
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:00<00:00, 193.18it/s, Materializing param=model.norm.weight]
trainable params: 5636096 || all params: 754911232 || trainables%: 0.746590560729662
[-] Running trainer...
[-] Model: meta-llama/Llama-3.2-1B
[-] Learning rate: 0.0003
[-] Batch size: 128
[-] Accumulation steps: 1
[!] Setting checkpoint save path to: all_Llama-3.2-1B
[!] Setting outputs log to: outputs_20260318-212854_Llama-3.2-1B.log
[+] Training in combined mode.
[!] SDD constraints: 2385 total (161 merged from augmented rules)
[-] Evaluation @ epoch 0...

[-] Computing scores on calibration split in val mode.
[-] Using prompt format: 0

[-] Querying facts for prompt 0...
100% 1/1 [00:01<00:00,  1.70s/it]

[-] Distribution:
# "true": 13
# total: 83

[val-calibration] Scoring factuality...
negation consistency; applicable: 83; violated: 66

[val-calibration] S

In [6]:
# Hybrid training
!python run.py --config configs/train_llama32_1b_hybrid.json

[-] Running in single instance...
[-] Loading decoder only model with quantization
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:00<00:00, 184.42it/s, Materializing param=model.norm.weight]
trainable params: 5636096 || all params: 754911232 || trainables%: 0.746590560729662
[-] Running trainer...
[-] Model: meta-llama/Llama-3.2-1B
[-] Learning rate: 0.0001
[-] Batch size: 128
[-] Accumulation steps: 1
[!] Setting checkpoint save path to: hybrid_Llama-3.2-1B
[!] Setting outputs log to: outputs_20260318-133301_Llama-3.2-1B.log
[+] Training in combined mode.
[!] SDD constraints: 2385 total (161 merged from augmented rules)
[!] Hybrid: 1029 complex rules for LTN, 2313 binary rules handled by SDD
[!] Subsampled training rules: 2000 / 7203
[-] Evaluation @ epoch 0...

[-] Computing scores on calibration split in val mode.
[-] Using prompt format: 0

[-] Querying facts for prompt 0...
  0% 0/1 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:12800

In [18]:
# Base model
!python run.py --config configs/eval_llama32_1b_base.json

[-] Running in single instance...
[-] Loading decoder only model with quantization
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:00<00:00, 187.10it/s, Materializing param=model.norm.weight]
trainable params: 5636096 || all params: 754911232 || trainables%: 0.746590560729662
[-] Running trainer...
[-] Model: meta-llama/Llama-3.2-1B
[-] Learning rate: 0.0003
[-] Batch size: 8
[-] Accumulation steps: 1
[!] Setting checkpoint save path to: ltn_Llama-3.2-1B
[!] Setting outputs log to: outputs_20260318-203122_Llama-3.2-1B.log
[+] Evaluating model.
[!] Subsampled silver eval facts: 30 entities, 4387 facts / 12636
[!] Subsampled silver test facts: 30 entities, 1102 facts / 3132
[!] Subsampled silver train facts: 30 entities, 3018 facts / 8554

[-] Computing scores on calibration split in test mode.
[-] Using prompt format: 0

[-] Querying facts for prompt 0...
100% 134/134 [02:11<00:00,  1.02it/s]

[-] Distribution:
# "true": 168
# total: 1072

[test-calib

In [16]:
# LTN
!python run.py --config configs/eval_llama32_1b_ltn.json --checkpoint checkpoints/epoch-2-ltn_Llama-3.2-1B

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: 3
wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /content/CS273P-LTN-LLM/wandb/offline-run-20260318_190420-ryxgp01k
[-] Running in single instance...
[-] Loading decoder only model with quantization
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:00<00:00, 180.36it/s, Materializing param=model.norm.weight]
trainable params: 5636096 || all params: 754911232 || trainables%: 0.746590560729662
[-] Loading LoRA adapter: checkpoints/epoch-2-ltn_Llama-3.2-1B
[-] Running trainer...
[-] Model: meta

In [ ]:
# SDD
!python run.py --config configs/eval_llama32_1b_sdd.json --checkpoint checkpoints/epoch-2-sdd_Llama-3.2-1B

In [17]:
# Hybrid
!python run.py --config configs/eval_llama32_1b_hybrid.json --checkpoint checkpoints/epoch-2-hybrid_Llama-3.2-1B

[-] Running in single instance...
[-] Loading decoder only model with quantization
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:00<00:00, 196.51it/s, Materializing param=model.norm.weight]
trainable params: 5636096 || all params: 754911232 || trainables%: 0.746590560729662
[-] Loading LoRA adapter: checkpoints/epoch-2-hybrid_Llama-3.2-1B
[-] Running trainer...
[-] Model: meta-llama/Llama-3.2-1B
[-] Learning rate: 0.0003
[-] Batch size: 8
[-] Accumulation steps: 1
[!] Setting checkpoint save path to: hybrid_Llama-3.2-1B
[!] Setting outputs log to: outputs_20260318-194801_Llama-3.2-1B.log
[+] Evaluating model.
[!] SDD constraints: 2385 total (161 merged from augmented rules)
[!] Hybrid: 1029 complex rules for LTN, 2313 binary rules handled by SDD
[!] Subsampled silver eval facts: 30 entities, 4387 facts / 12636
[!] Subsampled silver test facts: 30 entities, 887 facts / 2497
[!] Subsampled silver train facts: 30 entities, 3214 facts / 9126

[-] Compu

In [7]:
!cp -r checkpoints/ /content/drive/MyDrive/CS273P-LTN-LLM/checkpoints/